> Notebook-friendly copy of `part-I/1.1-environment-and-data-types-environmental-solutions.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [ ]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle do not ship every package this notebook imports.
# Colab and Kaggle start in an empty working directory.
# This is a no-op in an environment that is already set up.
import importlib.util
import subprocess
import sys

for module, package in {"pooch": "pooch"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

from pathlib import Path

Path("_files").mkdir(exist_ok=True)   # the folder this notebook writes into

# Environmental Solutions

**ℹ️ Reference solutions**

Worked solutions for the real-dataset exercise in [1.1-environment-and-data-types-environmental-exercises.ipynb](1.1-environment-and-data-types-environmental-exercises.ipynb).

## Exercise 1: A year of station data

The practice exercises used values typed by hand. This one uses a real record: daily maximum
temperatures for 2022 from the automated weather station at Independence Municipal Airport,
Iowa, US, station code IIB. The file has one row for each of the 365 days, but 16 of them, from
25 October to 9 November, have no temperature value. Work through the steps in order, since each
step reuses variables from the one before. The same file returns in 1.2, where you turn the whole
year into a monthly summary.

The data has three columns: the station code, the day as `dd.mm.yy`, and the daily maximum
temperature **in degrees Fahrenheit**. Converting to SI units is part of the job.

In [ ]:
# Pre-supplied: download the data file and cache it locally.
# You do not need to understand this cell yet — fetching data is covered in the
# reproducible-data-pipelines bonus subchapter.
import pooch

data_file = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/station_iib_daily_max_temp_2022.csv",
    known_hash="sha256:034755fb289b4e157e5d029995481786a359eb25a80df62c09ee83d063013144",
    fname="station_iib_daily_max_temp_2022.csv",
    path=pooch.os_cache("mlees"),
)
print("data cached at:", data_file)

**Step 1.** Open the file with a `with` block in read mode and read all its lines with
`.readlines()`.

Print how many lines the file has, then print the first three lines exactly as
they come off disk.

In [ ]:
from pathlib import Path

with Path(data_file).open("r", encoding="utf-8") as fhandler:
    lines = fhandler.readlines()

print("lines in file:", len(lines))      # 366: one header plus 365 days

# each line still ends in "\n", so print leaves a blank line after it
print(lines[0])                          # station,day,max_temp_f
print(lines[1])                          # IIB,01.01.22,14.7
print(lines[2])                          # IIB,02.01.22,8.6

**Step 2.** Look at the first record (the second line of the file). Split it on the separator
and print the resulting list. Then print the *type* of each of the three fields.

Answer in a comment: are the temperatures numbers at this point?

In [ ]:
first_record = lines[1].strip()      # line 0 is the header
fields = first_record.split(",")
print(fields)                        # ['IIB', '01.01.22', '14.7']

print(fields[0], "->", type(fields[0]))
print(fields[1], "->", type(fields[1]))
print(fields[2], "->", type(fields[2]))

# No: every field is a str at this point. Reading a file gives text, and the
# temperature only becomes a number once we cast it ourselves.

**Step 3.** Take that same first record and turn it into three properly typed, well-named
variables: the station code as text, the day as text, and the maximum temperature as a
**float in degrees Celsius**. Print them in one line with an f-string, showing the temperature
to one decimal with its unit.

Recall that °C = (°F − 32) × 5/9.

In [ ]:
station_code = fields[0]
day_1 = fields[1]
max_temp_fahrenheit_1 = float(fields[2])
max_temp_celsius_1 = (max_temp_fahrenheit_1 - 32.0) * 5.0 / 9.0

print(f"station {station_code} on {day_1}: {max_temp_celsius_1:.1f} °C")
print(type(station_code), type(day_1), type(max_temp_celsius_1))

**Step 4.** Now do the same for the *second* record (the third line of the file), so you have
two days side by side.

1. Ask which of the two days was warmer by comparing the two raw temperature **fields**, before
   any casting.
2. Ask the same question again, this time comparing the two values **after** casting to float.
3. The two answers disagree. In a comment, explain which one is right and why the other one is
   wrong.

In [ ]:
second_record = lines[2].strip()
fields_2 = second_record.split(",")
day_2 = fields_2[1]
max_temp_fahrenheit_2 = float(fields_2[2])
max_temp_celsius_2 = (max_temp_fahrenheit_2 - 32.0) * 5.0 / 9.0

In [ ]:
print(f"station {station_code} on {day_2}: {max_temp_celsius_2:.1f} °C")

In [ ]:
# 1. comparing the raw fields, before any casting
print("raw text says day 2 was warmer:", fields_2[2] > fields[2])

In [ ]:
# 2. comparing the cast values
print("numbers say day 2 was warmer:", max_temp_celsius_2 > max_temp_celsius_1)

# 3. The numbers are right: 8.6 °F is colder than 14.7 °F, so day 2 was not warmer.
#    The text comparison walks character by character and stops at the first one that
#    differs: '8' comes after '1', so it calls "8.6" the larger string. It never sees
#    that "14.7" has two digits before the decimal point and "8.6" has one.

**Step 5.** Write your own summary file, in SI units this time.

1. Build a path to `_files/station_iib_summary.csv` with `pathlib.Path`, creating the folder if
   it does not exist.
2. In one `with` block in mode `"w"`, write a header line `station,day,max_temp_celsius` and
   then one line for each of the two days you converted, with the temperature formatted to one
   decimal.
3. Reopen the file in mode `"r"`, read it back with `.read()`, and print it.

In [ ]:
from pathlib import Path

summary_path = Path("_files/station_iib_summary.csv")
summary_path.parent.mkdir(exist_ok=True)

with summary_path.open("w", encoding="utf-8") as fhandler:
    fhandler.write("station,day,max_temp_celsius\n")
    fhandler.write(f"{station_code},{day_1},{max_temp_celsius_1:.1f}\n")
    fhandler.write(f"{station_code},{day_2},{max_temp_celsius_2:.1f}\n")

with summary_path.open("r", encoding="utf-8") as fhandler:
    print(fhandler.read())